# Sarnia Air Quality, Traffic, and Weather Analysis

Comprehensive analysis for Sarnia including:
1. **Air Quality Data**: Station 14111 (Sarnia East)
2. **Traffic Data**: Highway 7 (Traffic ID: 3553005, Coordinates: -80.96954, 46.43512)
3. **Weather Data**: Sarnia weather station
4. **Analysis**: Correlation between traffic volume and air quality, controlled for weather effects

In [5]:
!pip install ipykernel
import pandas as pd
import glob
import os
import numpy as np
from pathlib import Path
import re
import seaborn as sns
import matplotlib.pyplot as plt


zsh:1: command not found: pip


## 1. Load Sarnia Air Quality Data (Station 14111)

In [6]:
# Load Sarnia Air Quality Data
AQ_FILE = "../data/air_quality/aq_data/station_14111_data.csv"

def load_sarnia_aq(file_path=AQ_FILE):
    """Load and process Sarnia air quality data from station 14111"""
    
    # Extract metadata from the file
    try:
        with open(file_path, 'r', encoding='latin1') as f:
            meta_lines = [next(f) for _ in range(10)]
            station_name = meta_lines[1].split(',')[1].strip()
            lat = meta_lines[3].split(',')[1].strip()
            lon = meta_lines[4].split(',')[1].strip()
    except Exception as e:
        print(f"Warning: Could not extract metadata: {e}")
        station_name = "Sarnia East"
        lat, lon = "42.98", "-82.39"
    
    # Load data
    df = pd.read_csv(file_path, skiprows=10, index_col=False, encoding='latin1')
    
    # Add metadata
    df['Station'] = station_name
    df['Latitude'] = lat
    df['Longitude'] = lon
    
    # Clean data
    df.replace([9999, -999, 9999.0, -999.0], np.nan, inplace=True)
    df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
    df = df.dropna(subset=['Date'])
    df = df.sort_values('Date')
    
    # Calculate daily mean from hourly data
    potential_cols = [f'H{i}' for i in range(1, 25)]
    valid_cols = [c for c in potential_cols if c in df.columns]
    df['NO2_Mean'] = df[valid_cols].mean(axis=1)
    
    # Keep essential columns
    cols_to_keep = ['Date', 'Station', 'NO2_Mean', 'Latitude', 'Longitude']
    available_keep = [c for c in cols_to_keep if c in df.columns]
    df_final = df[available_keep].copy()
    
    return df_final

# Load and inspect
df_sarnia_aq = load_sarnia_aq()
print(f"✅ Sarnia Air Quality Data Loaded: {df_sarnia_aq.shape}")
print(f"   Date Range: {df_sarnia_aq['Date'].min().date()} to {df_sarnia_aq['Date'].max().date()}")
print(f"   Station: {df_sarnia_aq['Station'].iloc[0]}")
print(f"\nFirst few rows:")
print(df_sarnia_aq.head())

Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "/Users/sanikapoojary/Desktop/Traffic-and-Air-Quality-Analysis/.venv/lib/python3.9/site-packages/IPython/core/interactiveshell.py", line 3550, in run_code
  File "/var/folders/x7/pwd5syls7pgbpzbq8lxj1k780000gn/T/ipykernel_81728/1760508651.py", line 46, in <module>
    df_sarnia_aq = load_sarnia_aq()
  File "/var/folders/x7/pwd5syls7pgbpzbq8lxj1k780000gn/T/ipykernel_81728/1760508651.py", line 20, in load_sarnia_aq
    df = pd.read_csv(file_path, skiprows=10, index_col=False, encoding='latin1')
  File "/Users/sanikapoojary/Desktop/Traffic-and-Air-Quality-Analysis/.venv/lib/python3.9/site-packages/pandas/io/parsers/readers.py", line 1026, in read_csv
  File "/Users/sanikapoojary/Desktop/Traffic-and-Air-Quality-Analysis/.venv/lib/python3.9/site-packages/pandas/io/parsers/readers.py", line 620, in _read
  File "/Users/sanikapoojary/Desktop/Traffic-and-Air-Quality-Analysis/.venv/lib/python3.9/site-packages/pandas/io/parsers/readers.py", line 1620, in

## 2. Audit Air Quality Data - Check for Gaps and Duplicates

In [ ]:
# Check for duplicates
duplicates = df_sarnia_aq[df_sarnia_aq.duplicated(subset=['Date'], keep=False)]

if not duplicates.empty:
    print(f"⚠️ Found {len(duplicates)} duplicate date rows")
    print("Duplicate dates:")
    print(duplicates.sort_values('Date'))
else:
    print("✅ No duplicate dates found")

# Check for gaps
df_sarnia_aq = df_sarnia_aq.copy()
df_sarnia_aq['Date'] = pd.to_datetime(df_sarnia_aq['Date'])

start_date = df_sarnia_aq['Date'].min()
end_date = df_sarnia_aq['Date'].max()
full_range = pd.date_range(start=start_date, end=end_date, freq='D')
missing_dates = full_range.difference(df_sarnia_aq['Date'])

print(f"\n🔎 Air Quality Data Audit (Station 14111)")
print(f"   Range: {start_date.date()} to {end_date.date()}")
print(f"   Total Expected Days: {len(full_range)}")
print(f"   Actual Days Present: {len(df_sarnia_aq)}")
print(f"   Missing Days: {len(missing_dates)}")

if len(missing_dates) > 0:
    print(f"\n⚠️ Sample of missing dates:")
    print(missing_dates[:5].strftime('%Y-%m-%d').tolist())
else:
    print("\n✅ No missing dates! Time series is continuous.")

## 3. Load Sarnia Traffic Data (Highway 7)

In [ ]:
# Load traffic data and filter for Highway 7
TRAFFIC_FILE = "../data/traffic/tf-ft-eng.csv"

# Load all traffic data
df_traffic_all = pd.read_csv(TRAFFIC_FILE)
print(f"Total traffic records: {len(df_traffic_all)}")
print(f"Available columns: {df_traffic_all.shape[1]}")

# Filter for Highway 7 / Sarnia location
df_traffic_hwy7 = df_traffic_all[df_traffic_all['camera_road'].str.contains('HWY-7', na=False, case=False)].copy()

print(f"\n✅ Traffic Data for Highway 7:")
print(f"   Filtered records: {len(df_traffic_hwy7)}")
print(f"   Traffic ID: {SARNIA_CONFIG['traffic_id']}")
print(f"   Coordinates: {SARNIA_CONFIG['traffic_coords']}")

if len(df_traffic_hwy7) > 0:
    print(f"\n   Unique camera_road values:")
    print(f"   {df_traffic_hwy7['camera_road'].unique()}")
    
    # Show sample of columns
    print(f"\n   Sample columns: {list(df_traffic_hwy7.columns[:10])}")

## 4. Transform Traffic Data to Long Format

In [ ]:
# Transform traffic data from wide to long format
# Identify daily traffic columns (xYYYY_MM_DD format)
date_col_pattern = re.compile(r"^x\d{4}_\d{2}_\d{2}$")
traffic_date_cols = [c for c in df_traffic_hwy7.columns if date_col_pattern.match(str(c))]

print(f"Found {len(traffic_date_cols)} daily traffic columns")

# Transform to long format
traffic_long = []
for idx, row in df_traffic_hwy7.iterrows():
    camera_road = row['camera_road']
    for col in traffic_date_cols:
        traffic_value = row[col]
        if pd.notna(traffic_value):
            date_str = col[1:]  # Remove 'x' prefix
            date = pd.to_datetime(date_str, format='%Y_%m_%d')
            traffic_long.append({
                'Date': date,
                'camera_road': camera_road,
                'traffic_count': traffic_value
            })

df_sarnia_traffic = pd.DataFrame(traffic_long)
df_sarnia_traffic = df_sarnia_traffic.sort_values('Date')

print(f"\n✅ Traffic Data Transformed:")
print(f"   Records: {len(df_sarnia_traffic)}")
print(f"   Date Range: {df_sarnia_traffic['Date'].min().date()} to {df_sarnia_traffic['Date'].max().date()}")
print(f"   Unique Locations: {df_sarnia_traffic['camera_road'].nunique()}")
print(f"\n   Locations:")
for loc in df_sarnia_traffic['camera_road'].unique():
    print(f"   - {loc}")
    
print(f"\nFirst few rows:")
print(df_sarnia_traffic.head())

## 5. Audit Traffic Data - Check for Gaps

In [ ]:
# Audit traffic data for gaps
df_sarnia_traffic_audit = df_sarnia_traffic.copy()
df_sarnia_traffic_audit['Date'] = pd.to_datetime(df_sarnia_traffic_audit['Date'])

start_date_t = df_sarnia_traffic_audit['Date'].min()
end_date_t = df_sarnia_traffic_audit['Date'].max()
full_range_t = pd.date_range(start=start_date_t, end=end_date_t, freq='D')
missing_dates_t = full_range_t.difference(df_sarnia_traffic_audit['Date'])

print(f"🔎 Traffic Data Audit (Highway 7 - Sarnia)")
print(f"   Range: {start_date_t.date()} to {end_date_t.date()}")
print(f"   Total Expected Days: {len(full_range_t)}")
print(f"   Actual Days Present: {len(df_sarnia_traffic_audit)}")
print(f"   Missing Days: {len(missing_dates_t)}")

if len(missing_dates_t) > 0:
    print(f"\n⚠️ Sample of missing dates:")
    print(missing_dates_t[:5].strftime('%Y-%m-%d').tolist())
    if len(missing_dates_t) > 5:
        print("...")
else:
    print("\n✅ No missing dates! Traffic data is continuous.")

## 6. Load Sarnia Weather Data

In [ ]:
# Load Sarnia Weather Data
WEATHER_FILE = "../data/weather/weather_data/Sarnia.csv"

WEATHER_COLS_TO_KEEP = {
    'Date/Time': 'Date',
    'Mean Temp (°C)': 'Mean_Temp',
    'Min Temp (°C)': 'Min_Temp',
    'Max Temp (°C)': 'Max_Temp',
    'Total Precip (mm)': 'Total_Precip',
    'Spd of Max Gust (km/h)': 'Max_Gust_Spd'
}

def load_sarnia_weather(file_path=WEATHER_FILE):
    """Load and process Sarnia weather data"""
    try:
        df = pd.read_csv(file_path)
        
        # Rename columns
        df = df.rename(columns=WEATHER_COLS_TO_KEEP)
        cols_to_select = [c for c in WEATHER_COLS_TO_KEEP.values() if c in df.columns]
        df = df[cols_to_select].copy()
        
        df['Date'] = pd.to_datetime(df['Date'])
        df = df.sort_values('Date')
        
        # Handle missing values - Forward Fill for continuous weather variables
        cols_to_fill = ['Mean_Temp', 'Min_Temp', 'Max_Temp', 'Total_Precip', 'Max_Gust_Spd']
        for col in cols_to_fill:
            if col in df.columns:
                df[col] = df[col].ffill()
        
        return df
    except Exception as e:
        print(f"Error loading weather data: {e}")
        return None

# Load weather data
df_sarnia_weather = load_sarnia_weather()

if df_sarnia_weather is not None and len(df_sarnia_weather) > 0:
    print(f"✅ Sarnia Weather Data Loaded: {df_sarnia_weather.shape}")
    print(f"   Date Range: {df_sarnia_weather['Date'].min().date()} to {df_sarnia_weather['Date'].max().date()}")
    print(f"   Columns: {list(df_sarnia_weather.columns)}")
    print(f"\nFirst few rows:")
    print(df_sarnia_weather.head())
else:
    print("⚠️ Could not load weather data. Continuing without weather variables.")

## 7. Merge All Datasets and Prepare for Analysis

In [ ]:
# Merge Air Quality and Traffic data
print("🔗 Merging Sarnia data (AQ + Traffic + Weather)...")

# Step 1: Merge AQ and Traffic
df_merged = pd.merge(
    df_sarnia_aq[['Date', 'NO2_Mean']], 
    df_sarnia_traffic[['Date', 'traffic_count']], 
    on='Date', 
    how='inner'
)

print(f"After AQ+Traffic merge: {df_merged.shape}")

# Step 2: Merge with Weather (if available)
if df_sarnia_weather is not None and len(df_sarnia_weather) > 0:
    weather_cols = [c for c in ['Date', 'Mean_Temp', 'Max_Gust_Spd', 'Total_Precip'] 
                    if c in df_sarnia_weather.columns]
    df_merged = pd.merge(
        df_merged,
        df_sarnia_weather[weather_cols],
        on='Date',
        how='inner'
    )
    print(f"After adding Weather: {df_merged.shape}")
else:
    print("⚠️ Weather data not available - proceeding with AQ + Traffic only")

# Clean the merged dataset
df_merged = df_merged.dropna()
df_merged = df_merged.sort_values('Date').reset_index(drop=True)

print(f"\n✅ Final Merged Dataset: {df_merged.shape}")
print(f"   Date Range: {df_merged['Date'].min().date()} to {df_merged['Date'].max().date()}")
print(f"   Columns: {list(df_merged.columns)}")
print(f"\nFirst few rows:")
print(df_merged.head())

# Display summary statistics
print(f"\nSummary Statistics:")
print(df_merged.describe())

## 8. Quick Correlation Analysis - Naive Model

In [ ]:
# Naive Correlation (Traffic vs NO2)
corr, p_value = pearsonr(df_merged['traffic_count'], df_merged['NO2_Mean'])

print("="*60)
print("🧪 NAIVE CORRELATION: Traffic vs Air Quality (NO2)")
print("="*60)
print(f"Pearson Correlation: {corr:.4f}")
print(f"P-Value: {p_value:.4e}")
print(f"Significant (p < 0.05): {'✅ YES' if p_value < 0.05 else '❌ NO'}")

# Visualize scatter plot with regression line
plt.figure(figsize=(10, 6))
sns.regplot(data=df_merged, x='traffic_count', y='NO2_Mean', 
            scatter_kws={'alpha':0.4, 's':20}, line_kws={'color':'red', 'linewidth':2})

plt.title(f"Sarnia: Traffic vs Air Quality (Pearson r={corr:.3f}, p={p_value:.2e})", 
          fontsize=14, fontweight='bold')
plt.xlabel("Daily Traffic Count", fontsize=12)
plt.ylabel("Daily NO2 Mean (ppb)", fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Standardize Data and Build Regression Models

In [ ]:
# Standardize data for regression modeling
# Z-scores allow us to compare relative effect sizes
df_scaled = df_merged.copy()
cols_to_scale = ['NO2_Mean', 'traffic_count']

# Add optional weather columns if available
if 'Mean_Temp' in df_scaled.columns:
    cols_to_scale.extend(['Mean_Temp', 'Max_Gust_Spd', 'Total_Precip'])

for col in cols_to_scale:
    if col in df_scaled.columns:
        df_scaled[col] = (df_scaled[col] - df_scaled[col].mean()) / df_scaled[col].std()

# Add weekend indicator
df_scaled['Date_Obj'] = pd.to_datetime(df_merged['Date'])
df_scaled['Is_Weekend'] = (df_scaled['Date_Obj'].dt.dayofweek >= 5).astype(int)

print("✅ Data standardized for regression modeling")
print(f"   Columns scaled: {cols_to_scale}")
print(f"   N records: {len(df_scaled)}")

# --- MODEL 1: Naive (Traffic Only) ---
print("\n" + "="*60)
print("MODEL 1: Naive (Traffic Only)")
print("="*60)

X1 = sm.add_constant(df_scaled[['traffic_count']])
y = df_scaled['NO2_Mean']
model_naive = sm.OLS(y, X1).fit()

print(f"Traffic Coefficient:  {model_naive.params['traffic_count']:.4f}")
print(f"Traffic P-Value:      {model_naive.pvalues['traffic_count']:.4f}")
print(f"R-Squared:            {model_naive.rsquared:.4f}")

# --- MODEL 2: Controlled (Traffic + Weather, if available) ---
print("\n" + "="*60)
print("MODEL 2: Weather-Controlled Model")
print("="*60)

if 'Mean_Temp' in df_scaled.columns:
    X2 = sm.add_constant(df_scaled[['traffic_count', 'Mean_Temp', 'Max_Gust_Spd', 'Total_Precip', 'Is_Weekend']])
    model_controlled = sm.OLS(y, X2).fit()
    
    print(f"Traffic Coefficient:  {model_controlled.params['traffic_count']:.4f}")
    print(f"Temperature Coeff:    {model_controlled.params['Mean_Temp']:.4f}")
    print(f"Wind Speed Coeff:     {model_controlled.params['Max_Gust_Spd']:.4f}")
    print(f"Precipitation Coeff:  {model_controlled.params['Total_Precip']:.4f}")
    print(f"Weekend Effect:       {model_controlled.params['Is_Weekend']:.4f}")
    print(f"Traffic P-Value:      {model_controlled.pvalues['traffic_count']:.4f}")
    print(f"R-Squared:            {model_controlled.rsquared:.4f}")
    
    print(f"\n📊 Model Comparison:")
    print(f"   Model 1 R²: {model_naive.rsquared:.4f}")
    print(f"   Model 2 R²: {model_controlled.rsquared:.4f}")
    print(f"   Improvement: {(model_controlled.rsquared - model_naive.rsquared):.4f}")
else:
    print("⚠️ Weather data not available for controlled model")
    model_controlled = None

## 10. Visualize Time Series Data

In [ ]:
# Plot 1: Air Quality Time Series
plt.figure(figsize=(14, 5))
plt.plot(df_merged['Date'], df_merged['NO2_Mean'], marker='o', markersize=2, linewidth=1, alpha=0.7, label='NO2')
plt.xlabel('Date', fontsize=12)
plt.ylabel('NO2 Concentration (ppb)', fontsize=12)
plt.title('Sarnia Air Quality Time Series (Station 14111)', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Plot 2: Traffic Time Series
plt.figure(figsize=(14, 5))
plt.plot(df_merged['Date'], df_merged['traffic_count'], marker='o', markersize=2, linewidth=1, alpha=0.7, color='orange', label='Traffic Count')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Daily Traffic Count', fontsize=12)
plt.title('Sarnia Traffic Time Series (Highway 7)', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Plot 3: Dual Axis (Normalized)
fig, ax1 = plt.subplots(figsize=(14, 6))

# Normalize for comparison
no2_norm = (df_merged['NO2_Mean'] - df_merged['NO2_Mean'].min()) / (df_merged['NO2_Mean'].max() - df_merged['NO2_Mean'].min())
traffic_norm = (df_merged['traffic_count'] - df_merged['traffic_count'].min()) / (df_merged['traffic_count'].max() - df_merged['traffic_count'].min())

ax1.plot(df_merged['Date'], no2_norm, 'b-', alpha=0.7, label='NO2 (Normalized)', linewidth=2)
ax1.set_xlabel('Date', fontsize=12)
ax1.set_ylabel('Normalized NO2', fontsize=12, color='b')
ax1.tick_params(axis='y', labelcolor='b')

ax2 = ax1.twinx()
ax2.plot(df_merged['Date'], traffic_norm, 'r-', alpha=0.7, label='Traffic (Normalized)', linewidth=2)
ax2.set_ylabel('Normalized Traffic', fontsize=12, color='r')
ax2.tick_params(axis='y', labelcolor='r')

plt.title('Sarnia: NO2 vs Traffic Volume (Normalized)', fontsize=14, fontweight='bold')
fig.tight_layout()
plt.show()

## 11. Model Diagnostic Plots

In [ ]:
# Diagnostic plots for the naive model
print("Model 1 (Naive) Summary:")
print(model_naive.summary())

# Plot residuals
fig = plt.figure(figsize=(12, 8))
sm.graphics.plot_partregress_grid(model_naive, fig=fig)
plt.tight_layout()
plt.show()

# If controlled model exists, show its diagnostics
if model_controlled is not None:
    print("\n" + "="*60)
    print("Model 2 (Controlled) Summary:")
    print(model_controlled.summary())
    
    # Residual plot
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    
    # Residuals vs Fitted
    residuals = model_controlled.resid
    fitted = model_controlled.fittedvalues
    axes[0, 0].scatter(fitted, residuals, alpha=0.5)
    axes[0, 0].axhline(y=0, color='r', linestyle='--')
    axes[0, 0].set_xlabel('Fitted Values')
    axes[0, 0].set_ylabel('Residuals')
    axes[0, 0].set_title('Residuals vs Fitted')
    
    # Q-Q plot
    sm.qqplot(residuals, line='45', ax=axes[0, 1])
    axes[0, 1].set_title('Q-Q Plot')
    
    # Scale-Location
    axes[1, 0].scatter(fitted, np.sqrt(np.abs(residuals/residuals.std())), alpha=0.5)
    axes[1, 0].set_xlabel('Fitted Values')
    axes[1, 0].set_ylabel('√|Standardized Residuals|')
    axes[1, 0].set_title('Scale-Location')
    
    # Residuals vs Observation Order
    axes[1, 1].plot(residuals)
    axes[1, 1].axhline(y=0, color='r', linestyle='--')
    axes[1, 1].set_xlabel('Observation Order')
    axes[1, 1].set_ylabel('Residuals')
    axes[1, 1].set_title('Residuals vs Observation Order')
    
    plt.tight_layout()
    plt.show()

## 12. Summary Statistics and Key Findings

In [ ]:
print("="*70)
print("📊 SARNIA ANALYSIS - SUMMARY REPORT")
print("="*70)

print("\n🔍 DATA SUMMARY:")
print(f"   Study Period: {df_merged['Date'].min().date()} to {df_merged['Date'].max().date()}")
print(f"   Valid Records: {len(df_merged)}")
print(f"   Air Quality Station: 14111 (Sarnia East)")
print(f"   Traffic Location: Highway 7 (ID: 3553005)")
print(f"   Weather Station: Sarnia")

print("\n📈 VARIABLE STATISTICS:")
print(f"   NO2 Mean: {df_merged['NO2_Mean'].mean():.2f} ± {df_merged['NO2_Mean'].std():.2f} ppb")
print(f"   Traffic Count: {df_merged['traffic_count'].mean():.0f} ± {df_merged['traffic_count'].std():.0f} vehicles/day")

if 'Mean_Temp' in df_merged.columns:
    print(f"   Temperature: {df_merged['Mean_Temp'].mean():.1f} ± {df_merged['Mean_Temp'].std():.1f} °C")
    print(f"   Max Gust Speed: {df_merged['Max_Gust_Spd'].mean():.1f} ± {df_merged['Max_Gust_Spd'].std():.1f} km/h")
    print(f"   Precipitation: {df_merged['Total_Precip'].mean():.1f} ± {df_merged['Total_Precip'].std():.1f} mm")

print("\n🧪 NAIVE ANALYSIS (Traffic Only):")
print(f"   Correlation: {corr:.4f}")
print(f"   P-Value: {p_value:.4e} {'✅ Significant' if p_value < 0.05 else '❌ Not Significant'}")
print(f"   R²: {model_naive.rsquared:.4f}")
print(f"   Interpretation: {'Moderate to strong' if abs(corr) > 0.5 else 'Weak to moderate'} relationship")

print("\n🔬 CONTROLLED ANALYSIS (With Weather):")
if model_controlled is not None:
    t_coef = model_controlled.params['traffic_count']
    t_pval = model_controlled.pvalues['traffic_count']
    print(f"   Traffic Effect: {t_coef:.4f} (p={t_pval:.4f}) {'✅' if t_pval < 0.05 else '❌'}")
    print(f"   Temperature Effect: {model_controlled.params['Mean_Temp']:.4f}")
    print(f"   Wind Effect: {model_controlled.params['Max_Gust_Spd']:.4f}")
    print(f"   Model R²: {model_controlled.rsquared:.4f}")
    
    improvement = model_controlled.rsquared - model_naive.rsquared
    print(f"\n   Model Improvement: {improvement:.4f} ({improvement*100:.2f}%)")
    
    if t_coef > 0 and t_pval < 0.05:
        print(f"\n   ✅ CONCLUSION: Traffic has a SIGNIFICANT positive effect on NO2")
    elif t_coef < 0 and t_pval < 0.05:
        print(f"\n   ✅ CONCLUSION: Traffic has a SIGNIFICANT negative effect on NO2 (Unexpected)")
    else:
        print(f"\n   ⚠️ CONCLUSION: Traffic effect is weak or confounded by weather")
else:
    print("   Not available - Weather data not loaded")

print("\n" + "="*70)
print("For detailed analysis, refer to the sections above")
print("="*70)

## 13. Export Results

In [ ]:
# Export the merged dataset for further analysis
output_file = "../data/sarnia_analysis_merged.csv"
df_merged.to_csv(output_file, index=False)
print(f"✅ Exported merged dataset to: {output_file}")

# Export the scaled dataset
scaled_output = "../data/sarnia_analysis_scaled.csv"
df_scaled.to_csv(scaled_output, index=False)
print(f"✅ Exported scaled dataset to: {scaled_output}")

# Create a summary report
report = f"""
SARNIA AIR QUALITY & TRAFFIC ANALYSIS REPORT
{'='*70}

ANALYSIS PERIOD:
  Start Date: {df_merged['Date'].min().date()}
  End Date: {df_merged['Date'].max().date()}
  Total Days: {len(df_merged)}

MONITORED LOCATIONS:
  Air Quality: Station 14111 (Sarnia East)
  Traffic: Highway 7 (Traffic ID: 3553005)
  Coordinates: -80.96954, 46.43512
  Weather: Sarnia Station

DATA SOURCES:
  Air Quality: station_14111_data.csv
  Traffic: tf-ft-eng.csv (filtered for HWY-7)
  Weather: Sarnia.csv

KEY STATISTICS:
  NO2 Mean: {df_merged['NO2_Mean'].mean():.2f} ± {df_merged['NO2_Mean'].std():.2f} ppb
  Traffic Mean: {df_merged['traffic_count'].mean():.0f} ± {df_merged['traffic_count'].std():.0f} vehicles/day

CORRELATION ANALYSIS:
  Pearson Correlation: {corr:.4f}
  P-Value: {p_value:.4e}
  Significant (p < 0.05): {'YES' if p_value < 0.05 else 'NO'}

MODEL 1 (NAIVE - TRAFFIC ONLY):
  R²: {model_naive.rsquared:.4f}
  Traffic Coefficient: {model_naive.params['traffic_count']:.4f}
  Traffic P-Value: {model_naive.pvalues['traffic_count']:.4f}

MODEL 2 (WEATHER-CONTROLLED):
"""

if model_controlled is not None:
    report += f"""  R²: {model_controlled.rsquared:.4f}
  Traffic Coefficient: {model_controlled.params['traffic_count']:.4f}
  Traffic P-Value: {model_controlled.pvalues['traffic_count']:.4f}
  Temperature Coefficient: {model_controlled.params['Mean_Temp']:.4f}
  Wind Speed Coefficient: {model_controlled.params['Max_Gust_Spd']:.4f}
  Model Improvement (R²): {model_controlled.rsquared - model_naive.rsquared:.4f}
"""
else:
    report += "  Not available (weather data unavailable)\n"

report += f"""
CONCLUSION:
  {'✅ Traffic has significant effect on air quality' if (model_naive.pvalues['traffic_count'] < 0.05) else '⚠️ Traffic effect is weak or confounded by weather'}

GENERATED: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}
"""

report_file = "../data/sarnia_analysis_report.txt"
with open(report_file, 'w') as f:
    f.write(report)
print(f"✅ Exported report to: {report_file}")

print("\n✅ Analysis complete! Files ready for further analysis.")

# Sarnia Air Quality and Traffic Analysis

This notebook provides a comprehensive analysis of air quality and traffic data for Sarnia, including:
1. Air quality measurements from monitoring station 14111
2. Traffic data for Highway 7
3. Weather data from Sarnia
4. Correlation analysis between traffic volume and air quality metrics